# Deep Joint Multi-Modal Yelp Fake Detector — PURE FULL (642k, no AI)

End-to-end network that fuses TWO modalities into one verdict:
* **Text branch** — GloVe + BiLSTM over the review text
* **Behavioural branch** — MLP over the 17 engineered reviewer/business features
* **Fusion head** — concatenate both branches → dense → sigmoid, trained **end-to-end**

**This variant trains on the PURE full Yelp dataset** (`yelp_multimodal_features_full.csv`,
678k reviews, the deduped YelpZip+NYC+Chi pool) — **no AI reviews**, the true human-fake task.
The train split is the real **~11% fake** distribution (val/test are balanced 50/50), so we
handle the imbalance with **balanced class weights** + a **val-tuned decision threshold**
(no AI ×10 up-weighting — that was the augmented/generalized objective). Deep counterpart to
the full-data pure sklearn Model B (xgb 77.3% / ens 76.2%).

**Before running:** upload `data/yelp_multimodal_features_full.csv` (~480 MB) to your Drive
root; set runtime to **GPU** (high-RAM recommended — 642k rows). **Output:** downloads
`yelp_multimodal_lstm_full_onnx.zip` (ONNX + tokenizer + scaler + threshold) → extract to
`data/yelp_multimodal_lstm_full_onnx/`. Feature order matches `model_b.FEAT_COLS` exactly, so
it is serving-compatible. Reports test macro-F1 + fake recall / precision / genuine specificity.

In [ ]:
# GloVe 300d (skip if already present)
import os
if not os.path.exists('glove.6B.300d.txt'):
    !wget -q http://nlp.stanford.edu/data/glove.6B.zip
    !unzip -q glove.6B.zip glove.6B.300d.txt
print('GloVe ready:', os.path.exists('glove.6B.300d.txt'))

In [ ]:
import numpy as np, pandas as pd, tensorflow as tf
from google.colab import drive
drive.mount('/content/drive')

CSV     = '/content/drive/MyDrive/yelp_multimodal_features_full.csv'   # PURE full 678k (no AI)
MAX_LEN = 250
VOCAB   = 30000
EMBED   = 300
DROP    = {'user_reviews_on_this_biz'}
META    = {'text','label','split','source','fake_type','generator','length_bucket'}

df = pd.read_csv(CSV)
df['text'] = df['text'].astype(str)
feat_cols = [c for c in df.columns if c not in META and c not in DROP]
print('rows', len(df), '| behavioural features', len(feat_cols))
# fake rate per split: train is the true ~11% (imbalanced), val/test balanced 50/50
print(df.groupby('split')['label'].agg(n='size', fake_rate='mean'))

In [ ]:
# splits
tr = df[df.split=='train'].reset_index(drop=True)
va = df[df.split=='val'].reset_index(drop=True)
te = df[df.split=='test'].reset_index(drop=True)
y_tr, y_va, y_te = tr.label.values, va.label.values, te.label.values

# --- TEXT modality: tokenize + pad ---
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
tok = Tokenizer(num_words=VOCAB, oov_token='<OOV>')
tok.fit_on_texts(tr.text.values)
def seq(s): return pad_sequences(tok.texts_to_sequences(s), maxlen=MAX_LEN, padding='post', truncating='post')
Xtr_t, Xva_t, Xte_t = seq(tr.text.values), seq(va.text.values), seq(te.text.values)

# --- BEHAVIOURAL modality: standardize (fit on train) ---
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
Xtr_b = sc.fit_transform(tr[feat_cols].values).astype('float32')
Xva_b = sc.transform(va[feat_cols].values).astype('float32')
Xte_b = sc.transform(te[feat_cols].values).astype('float32')
print('text', Xtr_t.shape, '| behavioural', Xtr_b.shape)

In [ ]:
# GloVe embedding matrix
emb_index = {}
with open('glove.6B.300d.txt', encoding='utf-8') as f:
    for line in f:
        v = line.split()
        emb_index[v[0]] = np.asarray(v[1:], dtype='float32')
emb_matrix = np.zeros((VOCAB, EMBED), dtype='float32')
hit = 0
for w, i in tok.word_index.items():
    if i < VOCAB:
        e = emb_index.get(w)
        if e is not None:
            emb_matrix[i] = e; hit += 1
print('GloVe coverage:', hit, '/', min(len(tok.word_index), VOCAB))

In [ ]:
# Deep joint multi-modal model (Keras functional API)
from tensorflow.keras.layers import (Input, Embedding, Bidirectional, LSTM,
    SpatialDropout1D, Dense, Dropout, Concatenate)
from tensorflow.keras.models import Model

# text branch
t_in = Input(shape=(MAX_LEN,), name='text_input')
x = Embedding(VOCAB, EMBED, weights=[emb_matrix], input_length=MAX_LEN, trainable=False)(t_in)
x = SpatialDropout1D(0.3)(x)
x = Bidirectional(LSTM(128))(x)
x = Dropout(0.4)(x)
text_repr = Dense(64, activation='relu')(x)

# behavioural branch
b_in = Input(shape=(len(feat_cols),), name='beh_input')
b = Dense(64, activation='relu')(b_in)
b = Dropout(0.3)(b)
beh_repr = Dense(32, activation='relu')(b)

# fusion head
f = Concatenate()([text_repr, beh_repr])
f = Dense(64, activation='relu')(f)
f = Dropout(0.4)(f)
out = Dense(1, activation='sigmoid')(f)

model = Model([t_in, b_in], out)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# Early stopping on VAL macro-F1 (custom; restores best weights)
from sklearn.metrics import f1_score
class MacroF1ES(tf.keras.callbacks.Callback):
    def __init__(self, vx, vy, patience=5):
        super().__init__(); self.vx=vx; self.vy=vy; self.patience=patience
        self.best=-1; self.best_w=None; self.best_thr=0.5; self.wait=0; self.best_ep=0; self.history=[]
    def on_epoch_end(self, epoch, logs=None):
        p = self.model.predict(self.vx, verbose=0).reshape(-1)
        bf, bt = -1, 0.5
        for t in np.linspace(0.05, 0.95, 19):
            s = f1_score(self.vy, (p>=t).astype(int), average='macro')
            if s>bf: bf, bt = s, t
        print('  epoch', epoch+1, 'val macroF1=', round(bf,4), '@thr', round(bt,2)); self.history.append(bf)
        if bf > self.best:
            self.best, self.best_thr = bf, bt
            self.best_w = self.model.get_weights(); self.wait = 0; self.best_ep = epoch+1
        else:
            self.wait += 1
            if self.wait >= self.patience:
                print('  early stopping (best epoch', self.best_ep, 'macroF1=', round(self.best,4), ')')
                self.model.stop_training = True
    def on_train_end(self, logs=None):
        if self.best_w is not None:
            self.model.set_weights(self.best_w)
            print('restored best epoch', self.best_ep, '(val macroF1=', round(self.best,4), 'thr=', round(self.best_thr,2), ')')

# CLASS IMBALANCE (pure full data): train is the true ~11% fake distribution (val/test balanced).
# No AI ×10 up-weighting here — instead use BALANCED class weights so the minority (fake) class
# isn't drowned out; the val-tuned threshold then sets the final operating point.
from sklearn.utils.class_weight import compute_class_weight
cw = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_tr)
class_weight = {0: float(cw[0]), 1: float(cw[1])}
print('class_weight (balanced):', class_weight)

es = MacroF1ES([Xva_t, Xva_b], y_va, patience=5)
hist = model.fit([Xtr_t, Xtr_b], y_tr, class_weight=class_weight,
                 validation_data=([Xva_t, Xva_b], y_va),
                 epochs=30, batch_size=256, callbacks=[es], verbose=1)

In [ ]:
# Training curves
import matplotlib.pyplot as plt
h=hist.history
fig,ax=plt.subplots(1,3,figsize=(16,4))
ax[0].plot(h['loss'],label='train'); ax[0].plot(h['val_loss'],label='val'); ax[0].set_title('Loss'); ax[0].set_xlabel('epoch'); ax[0].legend()
ax[1].plot(h['accuracy'],label='train'); ax[1].plot(h['val_accuracy'],label='val'); ax[1].set_title('Accuracy'); ax[1].set_xlabel('epoch'); ax[1].legend()
ax[2].plot(range(1,len(es.history)+1), es.history, marker='o', color='green', label='val macro-F1')
ax[2].axvline(es.best_ep, ls='--', color='red', label=f'best epoch {es.best_ep}')
ax[2].set_title('Val macro-F1'); ax[2].set_xlabel('epoch'); ax[2].legend()
plt.tight_layout(); plt.savefig('yelp_multimodal_lstm_training.png', dpi=120); plt.show()

In [ ]:
# Test evaluation (val-tuned threshold) — pure human-fake task, no per-AI-type breakdown
from sklearn.metrics import f1_score, accuracy_score, recall_score, precision_score, classification_report
thr  = es.best_thr
p_te = model.predict([Xte_t, Xte_b], verbose=0).reshape(-1)
pred = (p_te >= thr).astype(int)
print('threshold (val-tuned macroF1) =', round(thr, 2))
print('test accuracy  =', round(accuracy_score(y_te, pred), 4))
print('test macro-F1  =', round(f1_score(y_te, pred, average='macro'), 4))
print('fake recall    =', round(recall_score(y_te, pred, pos_label=1), 3))
print('fake precision =', round(precision_score(y_te, pred, pos_label=1), 3))
print('genuine spec   =', round(recall_score(y_te, pred, pos_label=0), 3))
print('\nReferences (same Yelp test): text-only ~69% · pure sklearn Model B (full) ens 76.2% / xgb 77.3%')
print()
print(classification_report(y_te, pred, target_names=['genuine','fake'], digits=4))

In [ ]:
# Save artifacts (keras for CPU re-export, + scaler/tokenizer/threshold for serving)
import json, joblib
THR=float(es.best_thr)
model.save('yelp_multimodal_lstm.keras')
joblib.dump(sc, 'yelp_multimodal_lstm_scaler.joblib')          # serving must standardise behaviour
with open('yelp_multimodal_lstm_tokenizer.json','w',encoding='utf-8') as f: f.write(tok.to_json())
json.dump({'threshold':THR,'feat_cols':feat_cols,'max_len':MAX_LEN,'vocab':VOCAB},
          open('yelp_multimodal_lstm_meta.json','w'), indent=2)
print('saved keras + scaler + tokenizer + meta | threshold', THR)

In [ ]:
# Export to ONNX (thread-safe serving, like the project's other DL models).
# Re-export the SavedModel with CUDA hidden so TF uses standard LSTM ops, not
# CudnnRNNV3 (which ONNX Runtime can't run). Multi-input: text_input + beh_input.
!pip install -q tf2onnx onnxruntime
import subprocess, os
open('cpu_export.py','w').write(
'''import os
os.environ["CUDA_VISIBLE_DEVICES"]=""
import tensorflow as tf
m=tf.keras.models.load_model("yelp_multimodal_lstm.keras")
m.export("yelp_multimodal_lstm_savedmodel")
print("CPU SavedModel export done")
''')
r1=subprocess.run(['python','cpu_export.py'],capture_output=True,text=True,
                  env={**os.environ,'CUDA_VISIBLE_DEVICES':''})
print(r1.stdout);  print('STDERR:',r1.stderr[-800:]) if r1.returncode else None
r2=subprocess.run(['python','-m','tf2onnx.convert','--saved-model','yelp_multimodal_lstm_savedmodel',
                   '--output','yelp_multimodal_lstm.onnx','--opset','13'],capture_output=True,text=True)
print(r2.stdout[-400:]); print('STDERR:',r2.stderr[-1200:]) if r2.returncode else None

In [ ]:
# Sanity check: ONNX outputs must match Keras (feed both inputs, mapped by last dim)
import onnxruntime as ort, numpy as np
sess=ort.InferenceSession('yelp_multimodal_lstm.onnx', providers=['CPUExecutionProvider'])
print('ONNX inputs:', [(i.name,i.shape) for i in sess.get_inputs()])
feed={}
for i in sess.get_inputs():
    feed[i.name]=(Xte_t[:4].astype(np.float32) if i.shape[-1]==MAX_LEN else Xte_b[:4].astype(np.float32))
onnx_p=sess.run(None, feed)[0].flatten()
keras_p=model.predict([Xte_t[:4],Xte_b[:4]],verbose=0).flatten()
print('ONNX :', np.round(onnx_p,4))
print('Keras:', np.round(keras_p,4))
print('match:', bool(np.allclose(onnx_p,keras_p,atol=1e-3)))

In [ ]:
# Bundle + download -> extract to data/yelp_multimodal_lstm_full_onnx/
import zipfile
from google.colab import files
with zipfile.ZipFile('yelp_multimodal_lstm_full_onnx.zip','w',zipfile.ZIP_DEFLATED) as zf:
    for fn in ['yelp_multimodal_lstm.onnx','yelp_multimodal_lstm_tokenizer.json',
               'yelp_multimodal_lstm_scaler.joblib','yelp_multimodal_lstm_meta.json']:
        zf.write(fn)
print('created yelp_multimodal_lstm_full_onnx.zip  (extract -> data/yelp_multimodal_lstm_full_onnx/)')
files.download('yelp_multimodal_lstm_full_onnx.zip')